# 实验 04 · 神经网络前向传播

目标：先用 NumPy 理解批量前向传播，再学习框架。

本实验为新增练习，仅依赖 NumPy；题干、测试均不包含参考实现。
从上到下执行，完成每个 TODO 后运行对应自测。

In [1]:
import numpy as np
rng = np.random.default_rng(42)

## 1 · 批量 Dense 层

实现 dense_batch(A,W,b,activation)，A 为 (m,n)，W 为 (n,k)，b 为 (k,)，activation 是可调用函数，返回 (m,k)。

In [2]:
def dense_batch(A, W, b, activation):

    return activation(A@W + b)

    raise NotImplementedError("TODO: 批量 Dense 层")

In [3]:
A = np.array([[1., 2.], [-1., 0.]])
W = np.array([[1., 0., -1.], [0., 2., 1.]])
b = np.array([0., -1., 0.])
out = dense_batch(A, W, b, lambda z: np.maximum(z, 0.))
assert out.shape == (2, 3)
np.testing.assert_allclose(out, [[1., 3., 1.], [0., 0., 1.]])
np.testing.assert_allclose(dense_batch(A, W, b, lambda z: z), [[1., 3., 1.], [-1., -1., 1.]])
print("通过：维度、激活函数、批量运算")

通过：维度、激活函数、批量运算


## 2 · 按行稳定 Softmax

实现 softmax_rows(Z)，Z 为 (m,k)，返回每行的概率分布。对绝对值很大的 logits 也应稳定。不要修改输入。

In [4]:
def softmax_rows(Z):

    Z_shifted = Z-np.max(Z, axis=1, keepdims=True)
    exp_Z = np.exp(Z_shifted)
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

    raise NotImplementedError("TODO: 稳定的批量 Softmax")

In [5]:
Z = np.array([[1000., 1001., 1002.], [-1000., -1000., -1000.]])
before = Z.copy()
with np.errstate(over='raise', invalid='raise', divide='raise'):
    P = softmax_rows(Z)
assert P.shape == Z.shape and np.isfinite(P).all()
assert (P >= 0).all() and (P <= 1).all()
np.testing.assert_allclose(P.sum(axis=1), [1., 1.])
np.testing.assert_allclose(P[0], [0.09003057, 0.24472847, 0.66524096], rtol=1e-6)
np.testing.assert_allclose(P[1], [1/3, 1/3, 1/3])
np.testing.assert_allclose(softmax_rows(Z + 12345.), P)
np.testing.assert_array_equal(Z, before)
print("通过：行归一化、已知概率、稳定性、平移不变性")

通过：行归一化、已知概率、稳定性、平移不变性


## 实验记录

写出输入/输出形状，解释一个边界情况，再记录一次修改参数前后的观察。

`dense_batch(A, W, b, activation)` 中 `A:(m,n)`、`W:(n,k)`、`b:(k,)`，输出 `(m,k)`；`softmax_rows(Z)` 输入 `Z:(m,k)`，输出 `(m,k)`，每行和为 1。

当 logits 很大（如 `1000,1001,1002`）时，直接 `exp` 会溢出；实现中先减去每行最大值再取指数，因此结果有限且满足平移不变性。

将 `b` 从 `[0,-1,0]` 改为 `[0,-2,0]` 后，第一样本第二列激活前值由 `3` 变为 `2`，ReLU 后由 `3` 变为 `2`；第二样本第二列由 `-1` 变为 `-2`，ReLU 后仍为 `0`。说明偏置会影响激活前输出，但 ReLU 会截断负值。